# 11 Failure Modes, Guardrails, and Evaluation

## 一个只会成功演示的 Agent 系统，不是系统，只是舞台效果

到这一章之前，整套项目已经把一条相对完整的理想路径走通了：模型能在控制面约束下给出结构化意图，MCP 能提供工具、资源和任务入口，本地模型能接进来，本地 server 能成为能力层，Runtime 能把它们组织成任务闭环，案例也能跑出一个对 HR 有意义的结果。

但如果作品停在这里，技术判断力其实还差最后一步。因为真正严肃的系统说明，不是只展示“在理想输入下可以工作”，而是要回答：**它会怎么失败，失败时系统是否知道自己在失败，以及怎样衡量它到底做得好不好。**

这也是 Agent 系统和普通演示脚本最重要的区别之一。脚本只要跑通 happy path 就够了，系统则必须知道边界在哪里、风险在哪里、恢复空间在哪里、评估标准在哪里。

## 先给结论

这一章最重要的判断可以压缩成一句话：

> Agent 系统的成熟度，不体现在它能不能跑出一次看起来不错的结果，而体现在它是否能识别失败类型、限制错误传播、记录运行轨迹，并用明确标准评估自己到底完成了什么。

这句话的含义很重。它意味着：

- 失败不是异常边角，而是系统设计的常态输入
- Guardrail 不是为了保守，而是为了让任务闭环保持可控
- Evaluation 不应只看最终回答好不好看，而应看整个执行链是否合理

也就是说，前面章节建立的是功能成立，这一章建立的是系统可信。

## 1. 为什么 Agent 的失败不是偶发，而是结构性的

普通问答系统的失败，很多时候表现为答案错了、说漏了、编造了。但 Agent 系统的失败会更复杂，因为它不仅要生成语言，还要维持目标、解释意图、调度能力、吸收反馈。这意味着失败可能发生在链路的任何一环。

一旦这样看，失败就不再是零散事故，而是结构性后果：

- 模型可能误判当前轮是否需要外部能力
- Runtime 可能把意图解释错
- Tool 参数可能结构合法但语义不够执行
- Resource 返回可能过长或过噪，污染后续推理
- 多轮状态可能逐渐偏离原目标

这就是为什么 Agent 不能只靠“模型整体挺聪明”来保障质量。它需要把失败看成系统设计对象，而不是把失败简单归咎于模型运气不好。

## 2. 一套像样的失败分类，应该从哪里开始

如果要管理失败，第一步不是写更多 if/else，而是先把失败分型。因为不同类型的失败，恢复方式完全不同。

对这套项目而言，至少可以把失败分成几类：

- `决策失败`：模型本该读 resource 或调 tool，却直接给出猜测
- `选择失败`：模型选错 tool、错读 resource、套错 prompt
- `参数失败`：结构看似合法，但值不够执行或粒度错误
- `结果失败`：tool / resource 返回过噪、为空、或与任务不匹配
- `状态失败`：Runtime 未正确更新任务状态，导致后续循环失真
- `终止失败`：系统该停不停，或该继续时过早结束

这套分类的意义不是为了把问题命名得更漂亮，而是为了让系统能根据失败类型采取不同对策。没有分类，所谓 guardrail 往往只会变成一堆泛化重试。

## 3. Tool 相关失败往往是最显眼的，但不一定是最根本的

在演示里，最容易被看见的失败往往是工具层：

- 调了错误的 tool
- 参数不完整
- 调用执行报错

这些失败当然重要，但如果只盯着它们，容易把问题误解成“工具接得不够严谨”。其实很多 tool 失败的根源来自更上游：

- system prompt 没有清楚规定何时必须先验证
- tool schema 描述不足，导致模型选择空间模糊
- Runtime 没有把 resource 路径开放出来，逼得模型过早选 tool

也就是说，tool failure 常常只是系统更深层设计问题的显性症状，而不是问题本身。

## 4. Resource 失败更隐蔽，因为它看起来不像“报错”

Resource 相关失败通常不像 tool error 那样直观。它们往往不会让系统立刻崩，而是悄悄污染后续判断。

典型情况包括：

- 读了不该读的资源，导致上下文跑偏
- 读了正确资源，但内容过长，关键控制信号被淹没
- 资源本身质量差，导致后续结论建立在噪音上
- Runtime 没有把 resource 的语义位置标清楚，模型分不清它是背景、证据还是草稿

这类失败危险的地方在于，它们很容易伪装成“模型发挥不稳”。实际上，模型后面产生的很多问题，是因为它在一段坏上下文里做了完全正常的续写。

## 5. State 失败是最容易被忽略、但最伤系统的一类问题

如果前面的章节强调了状态的重要性，那么这里就必须承认：状态更新失败几乎是 Agent 系统最伤的一类问题之一。

原因很直接。状态一旦失真，后面每一步可能都还“看起来合理”，但实际上是在沿着错误任务轨迹继续推进。

常见状态失败包括：

- 已确认事实没有被写回状态
- 已读资源没有被记录，导致重复读取
- 某步失败没有被标记，系统误以为已经成功推进
- 终止条件所依赖的计数或标志没有更新

这类问题之所以麻烦，是因为它们不一定表现为明显报错，而更像“系统越来越笨”。本质上，那不是模型变笨，而是 Runtime 的任务地图开始失真。

如果只是把失败命名出来，读者其实学不到什么。更有用的写法是直接说明：每一类失败在运行中长什么样。

`decision failure` 最常见的样子是，明明当前问题缺少关键事实，系统却直接给出一个看起来很完整的回答。读者看到的表象是“答得很顺”，但真正的问题是它跳过了 resource 或 tool 路径。`

`selection failure` 更像是系统知道自己要借外部能力，但借错了地方。比如本来应该先读岗位说明，却先去做匹配打分；或者本来应该调抽取工具，却套了一个只适合总结的 prompt。`

`argument failure` 则更具体一些：工具方向可能选对了，但参数不够执行。最典型的情况就是值看起来格式没错，实际上粒度不够，Runtime 接到之后还是没法真正往下跑。`

`state failure` 往往最隐蔽。系统表面上还在一轮轮推进，但它忘了自己已经读过什么、确认过什么、卡在哪里，于是开始重复读取、重复推断，或者拿着过期状态继续判断。`

`termination failure` 则发生在最后：不是系统完全不会结束，而是结束得不对。要么明明证据不够却收尾了，要么明明已经可以交付结果却还在继续绕圈。`

把失败这样写开之后，读者才能真正看到：这些问题不是同一种“模型不稳定”，而是不同层次的系统失配，因此后面的 guardrail 和恢复策略也不能一把梭。 

## 6. Guardrail 的真正作用，不是把模型绑死，而是阻止错误扩散

Guardrail 这个词很容易被理解成保守策略，仿佛它的作用只是限制模型自由发挥。这个理解不够准确。

在 Agent 系统里，guardrail 更重要的作用是：**阻止某一个局部错误，沿着多步链条不断放大。**

比如：

- 一次参数不完整，如果被直接执行，可能会把错误结果写回上下文
- 一次错误资源读取，如果不被标记，可能会污染接下来所有判断
- 一次错误终止，如果没有回退条件，可能让整个任务在假完成状态下结束

所以 guardrail 不是“让模型更听话”这么简单，而是让系统在出现局部偏差时还能守住任务边界。

## 7. 这套系统至少需要哪些 Guardrail

对当前项目来说，至少应该明确几类基础 guardrail：

- `结构 guardrail`：模型输出不符合预期结构时，拒绝直接执行
- `能力 guardrail`：模型请求不存在的 tool/resource/prompt 时，明确返回能力不可用
- `状态 guardrail`：每次动作后都必须有显式状态更新
- `终止 guardrail`：达到步数上限、失败阈值或无新信息阈值时必须收束
- `证据 guardrail`：关键结论应尽量能回溯到已读资源或已执行动作结果

这些 guardrail 不需要一开始做得复杂，但需要非常明确。因为一套没有 guardrail 的 Runtime，实际上是在默认相信每个环节都会稳定工作，这个前提在 LLM 系统里几乎永远不成立。

## 8. Retry 与 Fallback：恢复策略不该只有“再试一次”

很多系统遇到失败的第一反应就是重试。但在 Agent 场景里，盲目重试往往不是恢复，而只是延迟错误暴露。

更像样的恢复策略通常至少要区分：

- 如果是结构失败，也许应该要求重新生成结构化意图
- 如果是参数失败，也许应该回到澄清或参数补全路径
- 如果是资源噪音过高，也许应该先压缩再继续
- 如果是连续无进展，也许应该终止并给出原因说明

也就是说，retry 只是策略集合中的一种，而且往往不应是默认唯一策略。真正成熟的 Runtime 会根据失败类型选择不同恢复路径，而不是把一切都交给“模型再来一遍”。

## 9. 为什么 Agent 评估比普通回答评估难得多

普通问答系统的评估已经不简单，但 Agent 评估更难，因为它不只是一个最终答案问题，而是一个过程问题。

一个最终回答看起来不错的系统，可能实际上：

- 没真正使用到应使用的资源
- 调用了错误工具但碰巧蒙对结论
- 走了极度低效的路径才得到结果
- 在多个中间步骤已经偏离目标，但最后文风把问题盖过去了

因此，如果评估只看 final answer，就很容易把“会写”误判成“会执行”。而这个项目恰恰要避免这种误判。

## 10. 这套系统更合理的评估维度应该是什么

对当前项目而言，更合理的评估至少应该同时看几层：

- `结果质量`：最终输出是否满足任务目标
- `路径合理性`：是否读了该读的资源、调了该调的工具
- `结构稳定性`：模型意图是否稳定可解析
- `状态一致性`：Runtime 是否真的在推进而不是漂移
- `效率`：完成任务花了多少步、多少调用、多少冗余

这五层里，只有第一层比较接近普通 LLM 评估，其余几层都更像系统评估。也正因为如此，它们更能体现这套项目不是 prompt demo，而是 Agent 系统。

## 11. Observability 不是附加项，而是评估前提

如果没有 trace、没有中间状态、没有能力调用记录，那么很多评估维度根本无从谈起。你顶多只能评 final answer，而那正是最容易掩盖系统真实问题的层。

因此，一个像样的 Agent 系统要支持评估，先要支持观测。至少应该能看到：

- 每一步模型输出了什么意图
- Runtime 做了什么解释和分支判断
- 调了哪些能力对象，成功还是失败
- 状态在每一步发生了什么变化
- 最终是因为什么条件结束循环的

这些信息一旦缺失，所谓评估就会变成用最终文案倒推系统质量，这种评估几乎注定失真。

## 12. 对这套项目来说，什么才算“表现好”

为了避免评估永远停留在主观感受上，也有必要明确一下这套系统理想中的表现标准。一个“表现好”的运行轨迹，至少应该具备以下特征：

- 面对信息不足的任务时，倾向于先读 resource 或调用适当 tool，而不是直接猜
- 关键工具调用参数基本完整，不频繁走无意义重试
- 已获得的外部证据会影响后续输出，而不是形同虚设
- 结果里能看见证据组织，而不是单纯风格强势的断言
- 循环能在合理步数内终止，而不是靠强行截断

这些标准看起来朴素，但它们恰恰对应了 Agent 系统和普通生成系统之间最重要的差异。

## 13. 从展示角度看，为什么这一章特别重要

如果这个项目只是内部研发实验，那么失败模式和评估章节当然重要；但因为它同时是对外展示材料，这一章反而更不能缺。

原因在于，它能直接把项目从“会做 demo”提升到“会做系统”。因为面试官和技术经理通常很容易判断一个人会不会把 happy path 做漂亮，但更在意的是他是否知道：

- 系统最脆的地方在哪里
- 这些脆点应该如何被分型和约束
- 评估标准应该怎么建立，才不会被最终文风误导

把这些写出来，本质上是在展示你不仅能把东西做成，还知道它为什么会坏，以及怎么知道它坏了。

## 14. 为什么失败与评估之后，自然就该谈生产化

一旦失败模式、guardrails 和评估框架都被摆出来，下一步最自然的问题就不再是“这套系统能不能跑”，而是“如果想让它继续走向工程化，需要补哪些能力”。

那时讨论的重心会转向：

- 多个 server 怎么组织
- 权限和访问边界怎么治理
- prompt、tool、resource 怎么做版本化
- trace 和审计如何沉淀成运营能力

也就是说，失败与评估章节不是收尾，而是生产化章节的前置条件。因为没有这层理解，所谓 production 通常只是把 demo 部署得更正式而已。

## 15. 本章结论

这一章最值得保留的判断有这些：

- Agent 的失败是结构性的，不是偶发的。
- Guardrail 的核心作用不是限制模型，而是阻止错误在多步链路中扩散。
- 不同失败类型需要不同恢复路径，不能一律重试。
- Agent 评估不应只看 final answer，而应同时看路径、结构、状态和效率。
- 可观测性不是附加项，而是系统评估和调试的前提。

下一章会把视角再往前推一步：如果这套系统不再只是 notebook 原型，而是想成为可扩展工程，接下来在架构、治理和运行层面还需要补什么。